In [ ]:
"""Unit tests for PredictionPK model.

This suite focuses on smoke-testing the training step using a minimal
configuration tailored for PredictionPK. It uses a context split into
past/future and sets `n_of_target_individuals = 0` so the model only
operates on context data (as intended for PredictionPK).

Notes on shapes
- Batch `B=1` for quick tests; individuals kept small for speed.
- The data module returns a list of length `P` (number of permutations).
  We select the first element to obtain a single `AICMECompartmentsDataBatch`.
"""

from dataclasses import replace
from pathlib import Path

import pytest

from pff import config_dir
from pff.config_classes.node_pk_config import NodePKConfig
from pff.data.datasets.aicme_datasets import (
    AICMECompartmentsDataModule,
)
from pff.models.amortized_inference.prediction_pk import (
    PredictionForwardOutputs,
    PredictionPK,
)
from tests.helpers import DummyExperiment, DummyLogger, DummyTrainer
from tests.models.test_aicme_pk import _first_batch_list


def _prediction_config() -> NodePKConfig:
    """Return a minimal config for PredictionPK tests.

    - Uses past/future split in the context (default) so `context_rem_sim` is present.
    - Sets `n_of_target_individuals = 0` so batches contain only context series.
    - Keeps batch sizes small to execute quickly.
    """
    cfg = NodePKConfig()
    cfg.train = replace(
        cfg.train,
        batch_size=1,
        num_workers=0,
        persistent_workers=False,
        epochs=1,
    )
    cfg.mix_data = replace(
        cfg.mix_data,
        train_size=1,
        val_size=1,
        test_size=1,
        n_of_permutations=1,
        n_of_target_individuals=0,
    )
    cfg.meta_study = replace(cfg.meta_study, num_individuals_range=(2, 2))
    cfg.network = replace(cfg.network, aggregator_type="mean")

    # Context already defaults to divide past/future with remainder enabled.
    # We explicitly keep defaults to guarantee presence of `context_rem_sim`.
    return cfg

def _prediction_config_from_file() -> NodePKConfig:
    default_yaml = Path(config_dir) / "experiment_configs" / "node-pk" / "predictionPK.yaml"
    cfg: NodePKConfig = NodePKConfig.from_yaml(str(default_yaml))
    return cfg

In [4]:
cfg = _prediction_config_from_file()
dm = AICMECompartmentsDataModule(cfg)
batch_list = _first_batch_list(dm)
empirical_batches = dm.get_empirical_test_batches()
repo_id = "cesarali/lenuzza-2016"
empirical_batches = dm.get_empirical_test_batches()
if repo_id not in empirical_batches:
    pytest.skip(f"Empirical dataset '{repo_id}' is unavailable")
empirical_batch_list = empirical_batches[repo_id]

In [9]:
model = PredictionPK(cfg)
S = 5
batch = empirical_batch_list[0]
sampled, tgrid, real, mask = model.sample_individual_prediction(batch, sample_size=S)

In [12]:
mask.shape

torch.Size([18, 1, 12])